# SI4006 · Entrega M1 — Fine-tuning con LoRA (Encoder-Decoder)

**Tarea:** transformar diálogos médico-paciente (`dialogue`) en notas clínicas (`section_text`), usando el dataset MTS-Dialog.

---

Este notebook está basado en el lab de la semana 4 (`S04_Lab_Fine-tuning_LoRA.ipynb`), pero adaptado a un modelo
encoder-decoder. El input y el output son ambos texto libre, así que usamos un
modelo Seq2Seq (Flan-T5) en vez de un encoder de clasificación.

## 0 · Setup

Igual que en el lab: no fijamos `transformers`/`torch` (usamos los de Colab), solo instalamos lo que falta.
Agregamos `rouge_score` y `bert_score` porque son nuestras métricas principales (por ser una tarea de generación de texto y no clasificación como en el lad 4).

In [1]:
# Instalamos SOLO lo que falta. No fijamos transformers/torch (usamos los de Colab).
%pip install -q peft datasets evaluate accelerate rouge_score bert_score
# Quitamos el torchao viejo de Colab (choca con peft en get_peft_model; no lo usamos aquí).
%pip uninstall -y torchao
print('\nListo.')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.6 MB/s eta 0:00:00
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0

Listo.


In [2]:
import torch, transformers, peft
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('transformers', transformers.__version__, '| peft', peft.__version__)
print('torch', torch.__version__, '| device:', device)
if device == 'cpu':
    print('\n⚠️  Estás en CPU. Funciona, pero entrena lento. Activa la GPU T4 (ver arriba).')

transformers 5.13.1 | peft 0.19.1
torch 2.11.0+cu128 | device: cuda


## 1 · Los datos: MTS-Dialog

El dataset viene repartido en 4 archivos CSV en GitHub, usamos directamente la partición que el dataset ya trae:

- `MTS-Dialog-TrainingSet.csv` corresponde a  train
- `MTS-Dialog-ValidationSet.csv` corresponde a validation
- `MTS-Dialog-TestSet-1-MEDIQA-Chat-2023.csv` + `MTS-Dialog-TestSet-2-MEDIQA-Sum-2023.csv` los unimos en test
  (son dos sets de test con el mismo objetivo).

Cada fila tiene `ID`, `section_header`, `section_text` y `dialogue`. Nuestra tarea usa `dialogue` como input y
`section_text` como output; `section_header` no nos interesa y lo descartamos.

El resto de la descripción del dataset está en el markdown específico de la entrega para esto.

In [3]:
import pandas as pd

# Dataset
url_train = 'https://raw.githubusercontent.com/abachaa/MTS-Dialog/refs/heads/main/Main-Dataset/MTS-Dialog-TrainingSet.csv'
url_val   = 'https://raw.githubusercontent.com/abachaa/MTS-Dialog/refs/heads/main/Main-Dataset/MTS-Dialog-ValidationSet.csv'
url_test1 = 'https://raw.githubusercontent.com/abachaa/MTS-Dialog/refs/heads/main/Main-Dataset/MTS-Dialog-TestSet-1-MEDIQA-Chat-2023.csv'
url_test2 = 'https://raw.githubusercontent.com/abachaa/MTS-Dialog/refs/heads/main/Main-Dataset/MTS-Dialog-TestSet-2-MEDIQA-Sum-2023.csv'

cols = ['dialogue', 'section_text']

train_df = pd.read_csv(url_train)[cols].dropna().reset_index(drop=True)
val_df   = pd.read_csv(url_val)[cols].dropna().reset_index(drop=True)
test_df  = pd.concat([pd.read_csv(url_test1)[cols], pd.read_csv(url_test2)[cols]], ignore_index=True).dropna()
test_df  = test_df.reset_index(drop=True)

print('train:', train_df.shape)
print('validation:', val_df.shape)
print('test:', test_df.shape)

print('\nUn ejemplo real:')
print('  dialogue      :', train_df.loc[0, 'dialogue'][:200], '...')
print('  section_text  :', train_df.loc[0, 'section_text'][:200], '...')

train: (1201, 2)
validation: (100, 2)
test: (400, 2)

Un ejemplo real:
  dialogue      : Doctor: What brings you back into the clinic today, miss? 
Patient: I came in for a refill of my blood pressure medicine. 
Doctor: It looks like Doctor Kumar followed up with you last time regarding ...
  section_text  : The patient is a 76-year-old white female who presents to the clinic today originally for hypertension and a med check.  She has a history of hypertension, osteoarthritis, osteoporosis, hypothyroidism ...


## 2 · Modelo base + tokenizer, y tokenización

Cargamos Flan-T5-base (~250M, que es un encoder-decoder) con su tokenizer. Elegimos Flan-T5 en vez de un
T5 crudo porque, al estar afinado con instrucciones (instruction tuning), su baseline zero-shot (sin nuestro fine-tuning) ya produce
algo razonable cuando le damos una instrucción en texto, que es el objetivo del instruction tuning. Esto permite que el baseline es un punto de comparación honesta y no
solo ruido. Sigue siendo lo suficientemente pequeño para ser ejecutado en Colab.

Usamos el mismo prompt en el baseline y en el fine-tuning para que la comparación sea justa.

In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODELO = 'google/flan-t5-base'
tokenizer = AutoTokenizer.from_pretrained(MODELO)
model = AutoModelForSeq2SeqLM.from_pretrained(MODELO)
model.to(device)

PROMPT = 'Summarize the following doctor-patient dialogue into a clinical note:\n\n{dialogue}'

MAX_INPUT_LEN  = 512   # cubre p95 de la longitud de los diálogos (train)
MAX_TARGET_LEN = 200   # cubre p95 de la longitud de las notas clínicas de referencia

def tokenizar(batch):
    inputs = [PROMPT.format(dialogue=d) for d in batch['dialogue']]
    model_inputs = tokenizer(inputs, truncation=True, max_length=MAX_INPUT_LEN)
    labels = tokenizer(text_target=batch['section_text'], truncation=True, max_length=MAX_TARGET_LEN)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

print('Modelo y tokenizer listos.')

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Modelo y tokenizer listos.


In [5]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df).map(tokenizar, batched=True, remove_columns=['dialogue', 'section_text'])
val_ds   = Dataset.from_pandas(val_df).map(tokenizar, batched=True, remove_columns=['dialogue', 'section_text'])
test_ds  = Dataset.from_pandas(test_df).map(tokenizar, batched=True, remove_columns=['dialogue', 'section_text'])

print(train_ds)

Map:   0%|          | 0/1201 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1201
})


## 3 · El baseline: el mismo modelo sin fine-tuning (zero-shot)

El baseline es el modelo base sin fine-tuning, usando el mismo prompt que
se usará con el modelo tuneado. Medimos ROUGE y BERTScore sobre todo el conjunto de validación, para poder comparar
contra el modelo afinado sobre exactamente el mismo conjunto más adelante.

In [6]:
import evaluate
from tqdm import tqdm

rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")
model.eval()

baseline_df = val_df.copy()
references = baseline_df["section_text"].tolist()

baseline_predictions = []
BATCH = 8
dialogues = baseline_df["dialogue"].tolist()

for i in tqdm(range(0, len(dialogues), BATCH), desc="Generating baseline"):
    prompts = [PROMPT.format(dialogue=d) for d in dialogues[i:i + BATCH]]
    inp = tokenizer(prompts, return_tensors="pt", truncation=True,
                    max_length=MAX_INPUT_LEN, padding=True).to(device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=200, num_beams=4)
    baseline_predictions.extend(tokenizer.batch_decode(out, skip_special_tokens=True))

rouge_results = rouge.compute(predictions=baseline_predictions, references=references)
bert_results = bertscore.compute(predictions=baseline_predictions, references=references,
                                 lang="en", rescale_with_baseline=True)
bert_f1 = sum(bert_results["f1"]) / len(bert_results["f1"])
print(f"\nBASELINE — BERTScore\nF1: {bert_f1:.4f}")

print("BASELINE — ROUGE")
for metric, score in rouge_results.items():
    print(f"{metric}: {score:.4f}")

metrics = {
    **rouge_results,
    "bert_f1": bert_f1
}

pd.DataFrame([metrics]).to_csv("baseline_metrics.csv", index=False)

Generating baseline: 100%|██████████| 13/13 [01:02<00:00,  4.82s/it]


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



BASELINE — BERTScore
F1: 0.2923
BASELINE — ROUGE
rouge1: 0.2432
rouge2: 0.0894
rougeL: 0.2036
rougeLsum: 0.2038


## 4 · LoRA: fine-tuning eficiente

Igual que en el lab, congelamos el modelo y aprendemos solo las matrices pequeñas de LoRA. Hicimos los siguientes cambios a comparación del notebook de la semana 4:

- `task_type=TaskType.SEQ_2_SEQ_LM` (a diferencia del lab, que usaba `SEQ_CLS` para clasificación).
- `target_modules=['q', 'v']`: en la familia T5, las proyecciones de atención se llaman `q` y `v`
  (no `q_lin`/`v_lin` como en DistilBERT, ni `q_proj`/`v_proj` como en LLaMA/Qwen).
- `r=8`, `lora_alpha=16` (regla común `alpha ≈ 2·r`), igual que el lab y el paper original de LoRA.

In [7]:
from peft import LoraConfig, get_peft_model, TaskType

lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=['q', 'v'],   # capas de atención de T5 / Flan-T5
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()   # miren el % de parámetros que se entrena

trainable params: 884,736 || all params: 248,462,592 || trainable%: 0.3561


## 5 · Entrenar con la Seq2SeqTrainer API

Usamos `Seq2SeqTrainer` en vez de `Trainer` porque necesitamos `predict_with_generate=True`. Para tareas de
generación, la métrica se calcula sobre el texto generado, no sobre los logits crudos (a diferencia de
accuracy en el lab, que sí se calculaba directo sobre logits).

In [8]:
import numpy as np
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    return rouge.compute(predictions=decoded_preds, references=decoded_labels)

args = Seq2SeqTrainingArguments(
    output_dir='./m1_lora_out',
    learning_rate=2e-4,               # LoRA aguanta un LR más alto que el full fine-tuning
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy='epoch',
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    logging_steps=25,
    seed=42,
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)
trainer.train()

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,2.478127,2.226364,0.290712,0.109589,0.238324,0.239038
2,2.424601,2.181307,0.302891,0.113510,0.252446,0.253546
3,2.506467,2.170850,0.315080,0.115897,0.265034,0.266321


TrainOutput(global_step=453, training_loss=2.5161146412358906, metrics={'train_runtime': 612.7191, 'train_samples_per_second': 5.88, 'train_steps_per_second': 0.739, 'total_flos': 1853778385198080.0, 'train_loss': 2.5161146412358906, 'epoch': 3.0})

## 6 · Evaluar y comparar contra el baseline (validación)

Evaluamos el delta que aportó el fine-tuning sobre exactamente el mismo conjunto de validación que usamos en la Sección 3. Primero computamos las métricas para el modelo con fine-tuning.

In [9]:
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")
model.eval()

finetuned_df = val_df.copy()
references = finetuned_df["section_text"].tolist()

finetuned_predictions = []
BATCH = 8
dialogues = finetuned_df["dialogue"].tolist()

for i in tqdm(range(0, len(dialogues), BATCH), desc="Generating finetuned"):
    prompts = [PROMPT.format(dialogue=d) for d in dialogues[i:i + BATCH]]
    inp = tokenizer(prompts, return_tensors="pt", truncation=True,
                    max_length=MAX_INPUT_LEN, padding=True).to(device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=200, num_beams=4)
    finetuned_predictions.extend(tokenizer.batch_decode(out, skip_special_tokens=True))

rouge_results = rouge.compute(predictions=finetuned_predictions, references=references)
bert_results = bertscore.compute(predictions=finetuned_predictions, references=references,
                                 lang="en", rescale_with_baseline=True)
bert_f1 = sum(bert_results["f1"]) / len(bert_results["f1"])

print(f"\nFINETUNED — BERTScore\nF1: {bert_f1:.4f}")

print("FINETUNED — ROUGE")
for metric, score in rouge_results.items():
    print(f"{metric}: {score:.4f}")

metrics = {
    **rouge_results,
    "bert_f1": bert_f1
}

pd.DataFrame([metrics]).to_csv("finetuned_metrics.csv", index=False)

Generating finetuned: 100%|██████████| 13/13 [01:13<00:00,  5.69s/it]


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



FINETUNED — BERTScore
F1: 0.3750
FINETUNED — ROUGE
rouge1: 0.3368
rouge2: 0.1332
rougeL: 0.2822
rougeLsum: 0.2824


A continuación, computamos el delta y comparamos ambos modelos.


In [10]:
baseline = pd.read_csv("baseline_metrics.csv")
finetuned = pd.read_csv("finetuned_metrics.csv")

print("\nMETRICS COMPARISON")

for metric in ["rouge1", "rouge2", "rougeL", "rougeLsum", "bert_f1"]:
    base = baseline[metric].iloc[0]
    fine = finetuned[metric].iloc[0]
    delta = fine - base

    print(f"{metric:10s} | Baseline: {base:.4f} | Finetuned: {fine:.4f} | Δ: {delta:+.4f}")


METRICS COMPARISON
rouge1     | Baseline: 0.2432 | Finetuned: 0.3368 | Δ: +0.0936
rouge2     | Baseline: 0.0894 | Finetuned: 0.1332 | Δ: +0.0438
rougeL     | Baseline: 0.2036 | Finetuned: 0.2822 | Δ: +0.0786
rougeLsum  | Baseline: 0.2038 | Finetuned: 0.2824 | Δ: +0.0786
bert_f1    | Baseline: 0.2923 | Finetuned: 0.3750 | Δ: +0.0827


## 7 · Evaluación final sobre el conjunto de test

El conjunto de test no se tocó ni para entrenar ni para elegir
hiperparámetros, entonces es una medición honesta final del modelo fine-tuneado.

In [11]:
model.eval()

test_predictions = []

dialogues = test_df["dialogue"].tolist()
BATCH = 8

model.eval()

for i in tqdm(range(0, len(dialogues), BATCH), desc="Generating test"):
    prompts = [PROMPT.format(dialogue=d) for d in dialogues[i:i + BATCH]]

    inp = tokenizer(
        prompts,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LEN,
        padding=True
    ).to(device)

    with torch.no_grad():
        out = model.generate(
            **inp,
            max_new_tokens=200,
            num_beams=4
        )

    test_predictions.extend(
        tokenizer.batch_decode(out, skip_special_tokens=True)
    )

references = test_df["section_text"].tolist()

rouge_test = rouge.compute(
    predictions=test_predictions,
    references=references
)

bert_test = bertscore.compute(
    predictions=test_predictions,
    references=references,
    lang="en",
    rescale_with_baseline=True
)

bert_f1 = sum(bert_test["f1"]) / len(bert_test["f1"])

print("=== FINE-TUNED — TEST (held-out) ===")

print("\nROUGE")
for metrica, valor in rouge_test.items():
    print(f"{metrica}: {valor:.4f}")

print(f"\nBERTScore F1: {bert_f1:.4f}")

Generating test: 100%|██████████| 50/50 [05:40<00:00,  6.81s/it]


=== FINE-TUNED — TEST (held-out) ===

ROUGE
rouge1: 0.3229
rouge2: 0.1444
rougeL: 0.2731
rougeLsum: 0.2734

BERTScore F1: 0.3477


## 8 · Ejemplos cualitativos

Se piden al menos 3 ejemplos pero aquí mostramos 5. Se compara el input que recibe el modelo y se muestra el label que tiene el dataset original y luego se puede comparar el output del modelo zero-shot contra el fine-tuned. Esto permite una evaluación subjetiva del desempeño del modelo.

In [15]:
n_examples = 5
for i in range(n_examples):
    print('\n' + '=' * 80)
    print(f'EJEMPLO {i + 1}')
    print('=' * 80)
    print('\nDIÁLOGO:')
    print(val_df.loc[i, 'dialogue'][:600])
    print('\nREFERENCIA (nota clínica real):')
    print(val_df.loc[i, 'section_text'])
    print('\nBASELINE (zero-shot):')
    print(baseline_predictions[i])
    print('\nFINE-TUNED (LoRA):')
    print(finetuned_predictions[i])


EJEMPLO 1

DIÁLOGO:
Doctor: When did your pain begin? 
Patient: I've had low back pain for about eight years now.
Doctor: Is there any injury? 
 Patient: Yeah, it started when I fell in an A B C store.
Doctor: How old are you now?
Patient: I'm twenty six.  
Doctor: What kind of treatments have you had for this low back pain? 
Patient: Yeah, I got referred to P T, and I went, but only once or twice, um, and if I remember right, they only did the electrical stimulation, and heat. 
Doctor: I see, how has your pain progressed over the last eight years? 
Patient: It's been pretty continuous, but it's been at varying d

REFERENCIA (nota clínica real):
The patient is a 26-year-old female, referred to Physical Therapy for low back pain.  The patient has a history of traumatic injury to low back.  The patient stated initial injury occurred eight years ago, when she fell at a ABC Store.  The patient stated she received physical therapy, one to two visits and received modality treatment only, sp

## 9 · Guardar el adaptador LoRA

LoRA solo guarda las matrices pequeñas: son unos pocos MB, no el modelo entero.

In [13]:
model.save_pretrained('./m1_lora_adapter')
print('Adaptador LoRA guardado en ./m1_lora_adapter (solo los pesos de LoRA).')

Adaptador LoRA guardado en ./m1_lora_adapter (solo los pesos de LoRA).
